# Module 04 — UDF Performance

In [ ]:
import sys
import hashlib
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')
from pyspark.sql.functions import udf, pandas_udf, sha2, col
from pyspark.sql.types import StringType
import pandas as pd
from shared.spark_session import begin_module
from shared import config
from shared.utils import timed_action

spark = begin_module('module-04-udfs')
df = spark.read.parquet(*config.yellow_taxi_paths()).select('PULocationID').filter('PULocationID IS NOT NULL').limit(1_000_000)

## Python UDF

In [ ]:
@udf(StringType())
def py_hash(val):
    if val is None: return None
    return hashlib.md5(str(val).encode()).hexdigest()

result_py = df.withColumn('h', py_hash('PULocationID'))
_, t_py = timed_action(result_py)
print(f'Python UDF: {t_py:.2f}s')

## Pandas UDF (Arrow)

In [ ]:
@pandas_udf(StringType())
def pandas_hash(series: pd.Series) -> pd.Series:
    return series.astype(str).apply(lambda x: hashlib.md5(x.encode()).hexdigest())

result_pandas = df.withColumn('h', pandas_hash('PULocationID'))
_, t_pandas = timed_action(result_pandas)
print(f'Pandas UDF: {t_pandas:.2f}s')

## Native Spark

In [ ]:
result_native = df.withColumn('h', sha2(col('PULocationID').cast('string'), 256))
_, t_native = timed_action(result_native)
print(f'Native Spark: {t_native:.2f}s')
print(f'Speedup vs Python UDF — Pandas: {t_py/t_pandas:.1f}x, Native: {t_py/t_native:.1f}x')

## Key Takeaways

Prefer native Spark → Pandas UDF → Python UDF.